# Motor de observabilidade orientado a contratos

Materializa métricas em `case_nyc_taxi.governanca.obs_metricas` (tabela criada pelo `motores.py`, até agora vazia). Duas fontes, ambas genéricas — nenhuma lógica de dataset:

| Fonte | Métrica | Significado |
| --- | --- | --- |
| Tabelas dos contratos | `qtd_linhas` | contagem por tabela (bronze, silver, quarentena, gold) — `falha` se vazia |
| Tabelas dos contratos | `horas_desde_ultima_escrita` | freshness via `DESCRIBE HISTORY`; MVs/ST do UC usam o timestamp do último update da pipeline (fallback) |
| Event log SDP (`event_log()` TVF) | `expectativa[...]::passados` / `::reprovados` | registros que passaram/reprovaram cada expectativa no último update da pipeline — `reprovados > 0` = status `atencao` (inclui expectations warn por design) |

Toda a lista de tabelas vem dos YAML (`destinos.bronze/silver/quarentena` + gold `nome`/`agregados[].nome`). Observabilidade não interrompe o job — o gate de qualidade continua sendo o motor de `03_qualidade`. Widgets: `catalogo`, `pipeline_id` (o job do DAB passa `${resources.pipelines.case_nyc_taxi_medallion.id}`).

In [0]:
import os
import sys
from datetime import datetime, timezone

from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
candidatos = [Path(notebook_path), Path("/Workspace") / notebook_path.lstrip("/")]
REPO = None
for candidato in candidatos:
    for raiz in (candidato, *candidato.parents):
        if (raiz / "00_config" / "motores.py").is_file():
            REPO = str(raiz)
            break
    if REPO:
        break
if REPO is None:
    raise FileNotFoundError(f"Raiz do projeto nao localizada a partir do notebook: {notebook_path}")
sys.path.insert(0, str(Path(REPO) / "00_config"))
from motores import CONTRACTS_DIR, carregar_contrato, criar_tabelas_governanca, listar_contratos

# Widgets (execucao standalone ou via job do DAB)
dbutils.widgets.text("catalogo", "case_nyc_taxi")
dbutils.widgets.text("pipeline_id", "feda0232-181f-4451-b8d2-2f9aa024dd51")

catalogo = dbutils.widgets.get("catalogo")
pipeline_id = dbutils.widgets.get("pipeline_id")

criar_tabelas_governanca(catalogo, spark)
# naive UTC — DESCRIBE HISTORY e o event log devolvem datetime naive
COLETADO_EM = datetime.now(timezone.utc).replace(tzinfo=None)

# Tabelas monitoradas: derivadas dos contratos (sem logica de dataset).
# Ingestao -> bronze/silver/quarentena; gold -> consolidado + agregados.
tabelas = []
for contrato in listar_contratos():
    destinos = contrato.get("destinos", {})
    for camada in ("bronze", "silver", "quarentena"):
        if destinos.get(camada):
            tabelas.append((contrato["dataset"], camada, destinos[camada]))

for nome_arquivo in sorted(os.listdir(os.path.join(CONTRACTS_DIR, "gold"))):
    if nome_arquivo.endswith(".yml"):
        gold = carregar_contrato(f"gold/{nome_arquivo[:-4]}")
        tabelas.append((gold["nome"].split(".")[-1], "gold", gold["nome"]))
        for agregado in gold.get("agregados", []):
            tabelas.append((agregado["nome"].split(".")[-1], "gold", agregado["nome"]))

# Timestamp do ultimo update da pipeline (fallback de freshness para MVs/ST
# do UC, que nao aceitam DESCRIBE HISTORY)
ultimo_update = spark.sql(f"""
    SELECT timestamp FROM event_log('{pipeline_id}')
    WHERE event_type = 'create_update' ORDER BY timestamp DESC LIMIT 1
""").first()["timestamp"]

print(f"Pipeline: {pipeline_id}")
print(f"Ultimo update: {ultimo_update}")
print(f"Tabelas monitoradas: {len(tabelas)}")
for ds, camada, t in tabelas:
    print(f"  [{camada}] {t}")

In [0]:
metricas = []  # (dataset, tabela, metrica, valor, status)

for dataset, camada, tabela in tabelas:
    try:
        qtd = spark.read.table(tabela).count()
    except Exception as exc:
        print(f"[tabela ausente] {tabela}: {exc}")
        metricas.append((dataset, tabela, "qtd_linhas", None, "tabela_ausente"))
        continue

    try:
        historico = spark.sql(f"DESCRIBE HISTORY {tabela} LIMIT 1").collect()
        ultima_escrita = historico[0]["timestamp"] if historico else None
    except Exception:
        ultima_escrita = ultimo_update  # MV do UC: usa o último update da pipeline
    horas = (round((COLETADO_EM - ultima_escrita).total_seconds() / 3600, 1)
             if ultima_escrita else None)

    metricas.append((dataset, tabela, "qtd_linhas", float(qtd),
                     "ok" if qtd > 0 else "falha"))
    metricas.append((dataset, tabela, "horas_desde_ultima_escrita", horas, "ok"))
    print(f"[{camada}] {tabela}: {qtd:,} linhas | ultima escrita ha {horas}h".replace(",", "."))

In [0]:
# Expectativas do event log nativo do SDP, materializadas no ultimo update
df_expectativas = spark.sql(f"""
WITH latest AS (
    SELECT origin.update_id AS id
    FROM event_log('{pipeline_id}')
    WHERE event_type = 'create_update'
    ORDER BY timestamp DESC
    LIMIT 1
)
SELECT exp.dataset AS dataset,
       exp.name AS expectativa,
       CAST(SUM(exp.passed_records) AS DOUBLE) AS passados,
       CAST(SUM(exp.failed_records) AS DOUBLE) AS reprovados
FROM event_log('{pipeline_id}') e
JOIN latest ON e.origin.update_id = latest.id
LATERAL VIEW OUTER explode(
    from_json(e.details:flow_progress.data_quality.expectations,
              'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')
) t AS exp
WHERE e.event_type = 'flow_progress'
  AND exp.name IS NOT NULL
GROUP BY exp.dataset, exp.name
""")

linhas = df_expectativas.collect()
print(f"Expectativas no ultimo update: {len(linhas)}")
for r in linhas:
    # reprovados > 0 = "atencao": inclui expectations warn por design
    # (ex.: passenger_count nulo), desviadas para quarentena
    status = "ok" if (r["reprovados"] or 0) == 0 else "atencao"
    metricas.append((r["dataset"], "event_log",
                     f"expectativa[{r['expectativa']}]::passados", r["passados"], "ok"))
    metricas.append((r["dataset"], "event_log",
                     f"expectativa[{r['expectativa']}]::reprovados", r["reprovados"], status))
    print(f"  {r['dataset']}.{r['expectativa']}: {r['passados']} passaram / {r['reprovados']} reprovados")

In [0]:
df_metricas = spark.createDataFrame(
    [(ds, tb, m, v, st, COLETADO_EM) for ds, tb, m, v, st in metricas],
    "dataset STRING, tabela STRING, metrica STRING, valor DOUBLE, "
    "status STRING, coletado_em TIMESTAMP",
)
df_metricas.write.mode("append").saveAsTable(f"{catalogo}.governanca.obs_metricas")

atencao = [m for m in metricas if m[4] in ("falha", "tabela_ausente", "atencao")]
print(f"Observabilidade OK: {len(metricas)} metricas gravadas em "
      f"{catalogo}.governanca.obs_metricas")
if atencao:
    print(f"ATENCAO: {len(atencao)} metricas exigem atencao:")
    for ds, tb, m, v, st in atencao:
        print(f"  [{st}] {tb or ds} — {m} = {v}")
# Observabilidade nao interrompe o job: registra e reporta.
# O gate critico continua sendo o motor de qualidade (03_qualidade).